# Extração de Dados SRAG

Este notebook documenta a **ingestão** dos dados brutos do SRAG 2019–2026.

Fluxo:

```
Dados Abertos do SUS (Parquet)
        ↓
coletar_amostra()  → DataFrame
        ↓
CSV (intermediário legível)
        ↓
Parquet (formato final para análise)
```

> **Escopo:** aqui só se faz **extração e conversão**.
> Modelagem, validação e métricas ficam em notebooks posteriores.

## 1. Importações e configuração inicial

Funciona tanto se o notebook for executado da **raiz do projeto** quanto de dentro de `notebooks/`.

Aproveita `src.utils.env` para:
- carregar o `.env` (se existir);
- garantir que `data/raw`, `data/processed`, `data/treino` e `reports/figures` existam.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


# Raiz do projeto (funciona tanto da raiz quanto de notebooks/)
ROOT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

# Permite importar src
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


# Módulos internos
from src.utils import env
try:
    from src.data.extracao import coletar_amostra, csv_to_parquet  # type: ignore[attr-defined]
except ImportError:
    import importlib

    _extracao = importlib.import_module("src.data.extracao")
    coletar_amostra = _extracao.coletar_amostra
    csv_to_parquet = _extracao.csv_to_parquet


# Carrega .env (se existir) e garante diretórios padrão do projeto
env.carregar_dotenv()
env.garantir_diretorios()

print("ROOT_DIR:", ROOT_DIR)
print(env.resumo())

## 2. Diretório de dados brutos

In [ ]:
RAW_DIR = env.RAW
print(f"RAW_DIR: {RAW_DIR}")

## 3. Coleta dos dados brutos

⚠️ **Importante:** o SRAG 2019–2026 **não possui API paginada** nos Dados Abertos do SUS. O conjunto é disponibilizado apenas como **arquivo** (CSV, JSON, Parquet, XML).

Por isso, `coletar_amostra()` em `src/data/extracao.py` agora **baixa diretamente o arquivo Parquet** do portal e devolve um `DataFrame`:

```
https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026
        ↓
download do Parquet
        ↓
pd.read_parquet()
        ↓
DataFrame
```

Se o arquivo já existir localmente (`data/raw/<nome>.parquet`), ele é **reutilizado** — não baixa de novo.

In [ ]:
df = coletar_amostra()

print(f"\nTotal coletado: {len(df):,} registros")
print(f"Total de colunas: {len(df.columns)}")

if df.empty:
    raise ValueError("Nenhum registro foi coletado.")

df.head()

## 4. Persistência do CSV (intermediário legível)

In [ ]:
arquivo_csv = RAW_DIR / "srag_amostra.csv"

df.to_csv(
    arquivo_csv,
    index=False,
    encoding="utf-8-sig",
)

## 5. Verificação do total de registros salvos

In [ ]:
df_salvo = pd.read_csv(arquivo_csv, low_memory=False)

print(f"Arquivo salvo em: {arquivo_csv}")
print(f"Registros no DataFrame: {len(df):,}")
print(f"Registros no arquivo:   {len(df_salvo):,}")

if len(df) == len(df_salvo):
    print("✓ Verificação concluída: todos os registros foram salvos.")
else:
    print("⚠ Atenção: o número de registros não coincide.")

## 6. Conversão do CSV para Parquet

O Parquet é o formato final usado pelos notebooks de análise.
O nome `srag_amostra.parquet` evita confusão com o Parquet completo distribuído em `data/processed/dados-v1/srag.parquet`.

In [ ]:
PROCESSED_DIR = env.PROCESSED
arquivo_parquet = PROCESSED_DIR / "srag_amostra.parquet"

csv_to_parquet(
    csv_path=arquivo_csv,
    out_path=arquivo_parquet,
)

## 7. Verificação dos arquivos gerados

In [ ]:
print(f"CSV:     {arquivo_csv}")
print(f"Parquet: {arquivo_parquet}")

print(f"\nTamanho do CSV:     {arquivo_csv.stat().st_size / 1024**2:.2f} MB")
print(f"Tamanho do Parquet: {arquivo_parquet.stat().st_size / 1024**2:.2f} MB")